# 🌮 Tacos Las Cazuelitas — Análisis Exploratorio de Datos (EDA)
## Entregable E1/E2 · Cómputo en la Nube y Analítica de Datos (TecNM Campus Colima)

Este notebook documenta la exploración inicial de los pedidos sintéticos calibrados con la entrevista real a Doña Rocío de la Cruz:
- ~30 pedidos al día en promedio (martes a sábado).
- ~60 pedidos en días de quincena (1, 15, 16 y 30).
- Reparto de entrega: 10% llevar, 50% comer aquí, 40% domicilio (con incremento a domicilio en lluvia).
- Lengua se agota a las 10:00 a.m. (Pepena disponible toda la jornada).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

# Cargar dataset limpio
df = pd.read_csv("../data/clean/pedidos_clean.csv")
df["fecha_hora"] = pd.to_datetime(df["fecha_hora"])
df.info()

### 1. Estadísticas Descriptivas y Resumen Numérico

In [ ]:
display(df.describe())
print(f"Total pedidos únicos: {df['pedido_id'].nunique():,}")
print(f"Total líneas de detalle: {len(df):,}")

### 2. Distribución de Pedidos por Día de la Semana y Hora (Mapa de Calor)

In [ ]:
orden_dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
calor = df.groupby(["dia_semana", "hora"])["pedido_id"].nunique().reset_index(name="pedidos")
calor["dia_semana"] = pd.Categorical(calor["dia_semana"], categories=orden_dias, ordered=True)
matriz = calor.pivot_table(index="dia_semana", columns="hora", values="pedidos", fill_value=0)

plt.figure(figsize=(10, 5))
sns.heatmap(matriz, cmap="YlOrRd", annot=True, fmt="d", cbar_kws={"label": "Pedidos únicos"})
plt.title("Concentración Horaria de Pedidos Únicos por Día", fontsize=14, fontweight="bold")
plt.xlabel("Hora del día (7 a 12 h)")
plt.ylabel("Día de la semana")
plt.show()

### 3. Distribución del Modalidad de Entrega (Llevar / Aquí / Domicilio)

In [ ]:
pedidos_unicos = df.drop_duplicates(subset=["pedido_id"])
reparto = pedidos_unicos["tipo_entrega"].value_counts(normalize=True) * 100

plt.figure(figsize=(7, 4))
sns.barplot(x=reparto.index, y=reparto.values, palette="crest")
plt.title("Distribución Porcentual del Tipo de Entrega", fontsize=13, fontweight="bold")
plt.ylabel("Porcentaje (%)")
plt.xlabel("Modalidad")
for i, val in enumerate(reparto.values):
    plt.text(i, val + 1, f"{val:.1f} %", ha="center", fontweight="bold")
plt.ylim(0, 60)
plt.show()

### 4. Agotamiento de la Lengua a las 10:00 a.m. vs Pepena

In [ ]:
guisados = df[df["categoria"] == "guisado"].copy()
guisados["franja"] = guisados["hora"].apply(lambda h: "Antes de 10 a.m." if h < 10 else "10 a.m. a 12 h")

resumen = guisados.groupby(["producto", "franja"]).size().unstack(fill_value=0)
display(resumen.loc[["Pepena", "Lengua", "Chicharrón"]])

print("Obsérvese cómo la Lengua cae drásticamente después de las 10:00 h mientras Pepena y Chicharrón mantienen ventas estables.")